In [ ]:
!pip show sentence-transformers
!pip show faiss-cpu
!pip show PyPDF2
!pip show mistralai

Name: sentence-transformers
Version: 5.6.0
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 
Name: faiss-cpu
Version: 1.14.3
Summary: A library for efficient similarity search and clustering of dense vectors.
Home-page: https://github.com/facebookresearch/faiss
Author: Meta AI Research
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy, packaging
Required-by: 
Name: PyPDF2
Version: 3.0.1
Summary: A pure-python PDF library capable of splitting, merging, cropping, and transforming PDF files
Home-page: 
Author: 
Author-email: Mathieu Fenniak <biziqe@mathieu.fenniak.net>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 

In [ ]:
import PyPDF2
import numpy as np
import faiss
import os
import getpass
from sentence_transformers import SentenceTransformer

try:
    from mistralai import Mistral
except ImportError:
    from mistralai.client import Mistral

import textwrap

In [ ]:
def load_pdf(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text
pdf_path = "/content/Week7_Project (1).pdf"

In [ ]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end >= len(words):
            break
        start += chunk_size - overlap
    return chunks

dummy = "This is a test sentence. " * 100
test_chunks = chunk_text(dummy, chunk_size=50, overlap=10)
print(f"Number of chunks: {len(test_chunks)}")
print(f"First chunk word count: {len(test_chunks[0].split())}")

Number of chunks: 13
First chunk word count: 50


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_chunks(chunks):
    embeddings = embedding_model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
    return embeddings.astype("float32")

In [ ]:
class VectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatL2(dim)
        self.chunks = []
    def add(self, embeddings, chunks):
        self.index.add(embeddings)
        self.chunks.extend(chunks)
    def search(self, query_embedding, top_k=3):
        distances, indices = self.index.search(query_embedding, top_k)
        results = [self.chunks[i] for i in indices[0] if i < len(self.chunks)]
        return results

In [ ]:
def retrieve_context(query, vector_store, top_k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    return vector_store.search(query_embedding, top_k=top_k)

In [ ]:
os.environ["MISTRAL_API_KEY"] = getpass.getpass("Enter your Mistral API key: ")
mistral_client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
def generate_answer(query, context_chunks):
    context = "\n".join(context_chunks)
    prompt = f"""Answer the question using only the context below. If the answer isn't in the context, say you don't know.
Context:
{context}
Question: {query}
Answer:"""
    response = mistral_client.chat.complete(
        model="mistral-small-latest",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content

Enter your Mistral API key: ··········


In [ ]:
def build_rag_pipeline(pdf_path, chunk_size=150, overlap=30):
    print("Loading document...")
    text = load_pdf(pdf_path)
    print("Chunking text...")
    chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
    print(f"Created {len(chunks)} chunks")
    print("Creating embeddings...")
    embeddings = embed_chunks(chunks)
    print("Building vector store...")
    store = VectorStore(dim=embeddings.shape[1])
    store.add(embeddings, chunks)
    print("Pipeline ready!")
    return store
def ask(query, store, top_k=3):
    context_chunks = retrieve_context(query, store, top_k=top_k)
    answer = generate_answer(query, context_chunks)
    print("Question:", query)
    print("\nRetrieved context:")
    for i, c in enumerate(context_chunks):
        print(f"  [{i+1}] {textwrap.shorten(c, width=120)}")
    print("\nAnswer:", answer)
    return answer

In [ ]:
store = build_rag_pipeline(pdf_path)
ask("What are the main stages of the RAG pipeline?", store)

Loading document...
Chunking text...
Created 5 chunks
Creating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Building vector store...
Pipeline ready!
Question: What are the main stages of the RAG pipeline?

Retrieved context:
  [1] pipelines Conclusion This project demonstrates how to build a system that can understand user queries, retrieve [...]
  [2] Document Question Answering System (RAG) Dataset: Simple Beginner Dataset (Easiest) Use your own PDFs: ● Notes ● [...]
  [3] chunks into embeddings 4. Store embeddings in a vector database 5. Accept user query 6. Retrieve relevant chunks [...]

Answer: The main stages of the RAG pipeline, as described in the context, are:

1. **Chunking**: Splitting documents into smaller text chunks.
2. **Embedding**: Converting the text chunks into embeddings.
3. **Storing**: Storing the embeddings in a vector database.
4. **Querying**: Accepting a user query.
5. **Retrieval**: Retrieving relevant chunks based on the query.
6. **Generation**: Generating an answer using the retrieved context.


'The main stages of the RAG pipeline, as described in the context, are:\n\n1. **Chunking**: Splitting documents into smaller text chunks.\n2. **Embedding**: Converting the text chunks into embeddings.\n3. **Storing**: Storing the embeddings in a vector database.\n4. **Querying**: Accepting a user query.\n5. **Retrieval**: Retrieving relevant chunks based on the query.\n6. **Generation**: Generating an answer using the retrieved context.'

In [ ]:
ask("What embedding and vector store components does this project use?", store)

Question: What embedding and vector store components does this project use?

Retrieved context:
  [1] chunks into embeddings 4. Store embeddings in a vector database 5. Accept user query 6. Retrieve relevant chunks [...]
  [2] chunk is converted into a vector representation capturing its semantic meaning. 4. Vector Database Embeddings are [...]
  [3] PDFs or text files ● Learn how modern AI systems work internally Key Concepts 1. Retrieval Retrieval is [...]

Answer: The context does not specify the exact embedding model or vector store used in the project. You don't know.


"The context does not specify the exact embedding model or vector store used in the project. You don't know."